In [1]:
# import libraries
import pandas as pd
import numpy as np
import sys

sys.path.append('../')

from src.data_loader import load_cement_data



In [2]:
# load the cement data
df_raw = load_cement_data()
df = df_raw.copy()

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['site_id', 'date']).reset_index(drop=True)

df.head()

,date,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [3]:
# date based features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)


In [4]:
# England public holidays
holiday_dates = pd.to_datetime([
  # 2022
  '2022-01-03', '2022-04-15', '2022-04-18', '2022-05-02', 
  '2022-06-02', '2022-06-03', '2022-08-29', '2022-09-19',
  '2022-12-26', '2022-12-27',
  # 2023
  '2023-01-02', '2023-04-07', '2023-04-10', '2023-05-01', 
  '2023-05-08', '2023-05-29', '2023-08-28', '2023-12-25', 
  '2023-12-26',
  # 2024
  '2024-01-01', '2024-03-29', '2024-04-01', '2024-05-06', 
  '2024-05-27', '2024-08-26', '2024-12-25', '2024-12-26'
])

df['is_holiday'] = df['date'].isin(holiday_dates).astype(int)


In [8]:
# consumption lag features
lag_days = [1, 3, 7, 14, 28]

for lag in lag_days:
  df[f"lag_{lag}"] = (
df.groupby('site_id')['consumed_tonnes']
    .shift(lag)
  )

In [10]:
# rolling window moving average features
df['rolling_avg_7'] = (
  df.groupby('site_id')['consumed_tonnes']
  .shift(1)
  .rolling(window=7)
  .mean()
  .reset_index(level=0, drop=True)
)

df['rolling_avg_28'] = (
  df.groupby('site_id')['consumed_tonnes']
  .shift(1)
  .rolling(window=28)
  .mean()
  .reset_index(level=0, drop=True)
)

In [12]:
# inventory features - days of coverage
df['avg_daily_consumption'] = df["rolling_avg_28"]

df['days_of_coverage'] = (
  df['closing_inventory_tonnes'] / df['avg_daily_consumption']
)

df['days_of_coverage'] = df['days_of_coverage'].replace(
  [np.inf, -np.inf],
  np.nan
)

In [13]:
# weather features - 3 day rolling rain total, temperature trend
df['rain_3day_total'] = (
  df.groupby('site_id')['rain_mm']
  .shift(1)
  .rolling(window=3)
  .sum()
  .reset_index(level=0, drop=True)
)

df['temp_avg_3day'] = (
  df.groupby('site_id')['avg_temp_c']
  .shift(1)
  .rolling(window=3)
  .mean()
  .reset_index(level=0, drop=True)
)

df['temp_trend_3day'] = df['avg_temp_c'] - df['temp_avg_3day']

In [15]:
df.isnull().sum()

date                          0
site_id                       0
region                        0
behavior                      0
cement_type                   0
planned_pour_tonnes           0
consumed_tonnes               0
opening_inventory_tonnes      0
deliveries_tonnes             0
closing_inventory_tonnes      0
rain_mm                       0
avg_temp_c                    0
silo_capacity                 0
day_of_week                   0
month                         0
quarter                       0
is_weekend                    0
is_holiday                    0
lag_1                        30
lag_3                        90
lag_7                       210
lag_14                      420
lag_28                      840
rolling_avg_7               210
rolling_avg_28              840
avg_daily_consumption       840
days_of_coverage            840
rain_3day_total              90
temp_avg_3day                90
temp_trend_3day              90
dtype: int64

In [16]:
# drop rows with missing values created by lag and rolling features
df = df.dropna().reset_index(drop=True)

In [17]:
# define the features and target variable
features_cols = [
  "planned_pour_tonnes",
  "avg_temp_c",
  "rain_mm",
  "day_of_week",
  "month",
  "quarter",
  "is_weekend",
  "is_holiday",
  "lag_1",
  "lag_3",
  "lag_7",
  "lag_14",
  "lag_28",
  "rolling_avg_7",
  "rolling_avg_28",
  "days_of_coverage",
  "rain_3day_total",
  "temp_avg_3day",
  "temp_trend_3day"
]

target_col = "consumed_tonnes"

x = df[features_cols]
y = df[target_col]